# Using Whisper Off the Shelf

OpenAI's Whisper is a speech recognition model trained on 680,000 hours of multilingual audio. This notebook shows you how to load it from HuggingFace, run inference on audio files, interpret the output, and handle multiple languages. We use `openai/whisper-small` throughout, which is small enough to run on a laptop CPU while still being surprisingly capable.

In [ ]:
# pip install transformers torch soundfile librosa  # uncomment if needed
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import soundfile as sf
import librosa
from transformers import (
    pipeline,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

print(f"torch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. The Pipeline API: Quickest Path to Transcription

HuggingFace's `pipeline()` function wraps a model and its preprocessor into a single callable. For most use cases, this is all you need.

What happens under the hood when you call a speech pipeline:
1. Your audio is loaded and resampled to 16 kHz
2. Log-Mel spectrogram features are extracted (80 frequency bins, 25ms windows, 10ms stride)
3. The Whisper encoder processes the spectrogram into a sequence of hidden states
4. The Whisper decoder autoregressively generates text tokens
5. Tokens are decoded into a string

You don't need to handle any of that manually when using the pipeline.

In [ ]:
# Load the pipeline once; reuse it for all transcriptions
DEVICE = 0 if torch.cuda.is_available() else -1  # 0 = first GPU, -1 = CPU

asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-small",
    device=DEVICE,
)

print("Pipeline loaded.")
print(f"Model: {asr_pipeline.model.__class__.__name__}")
print(f"Params: {sum(p.numel() for p in asr_pipeline.model.parameters()) / 1e6:.1f}M")

### Full Whisper Inference with AutoModelForSpeechSeq2Seq

The `pipeline` helper is built on top of `AutoModelForSpeechSeq2Seq` and `AutoProcessor`. Using them directly gives you full control over generation parameters.

In [ ]:
# pip install transformers torch accelerate  # uncomment if needed
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline
import torch, time

def build_whisper_pipeline(model_id: str, device: str = None):
    """
    Build an ASR pipeline using AutoModelForSpeechSeq2Seq + AutoProcessor.
    This is the recommended HuggingFace pattern for Whisper inference.

    Args:
        model_id: e.g. "openai/whisper-base" or "openai/whisper-large-v3"
        device: "cuda", "cpu", or None for auto-detection

    Returns:
        HuggingFace pipeline callable
    """
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    print(f"Loading {model_id} on {device} ...")
    t0 = time.time()

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = hf_pipeline(
        task="automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
    )

    elapsed = time.time() - t0
    params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Loaded in {elapsed:.1f}s  |  {params:.0f}M parameters")
    return pipe


# Load whisper-base (good balance of speed and accuracy)
pipe_base = build_whisper_pipeline("openai/whisper-base")

# Uncomment to also load the large model (requires more VRAM/RAM):
# pipe_large = build_whisper_pipeline("openai/whisper-large-v3")

print("\nPipeline ready. Use pipe_base(audio_path) to transcribe.")

### Creating Test Audio

For this notebook we synthesize a test audio file (a pure tone) to avoid requiring downloads. In practice you'd load real `.wav` or `.mp3` files from disk.

## 1b. Audio Loading and Resampling with torchaudio

Whisper requires audio at 16 kHz mono. Real-world recordings often come at 44.1 kHz or 48 kHz, and may be stereo. This section shows the canonical pipeline using `torchaudio`.

In [ ]:
# pip install torchaudio matplotlib  # uncomment if needed
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

TARGET_SR = 16000  # Whisper's required sample rate

def load_and_resample(path: str, target_sr: int = TARGET_SR):
    """
    Load an audio file with torchaudio, resample to target_sr, convert to mono.

    Returns:
        waveform: torch.Tensor of shape [1, num_samples], float32
        sample_rate: int (will equal target_sr after resampling)
    """
    # torchaudio.load returns (waveform, sample_rate)
    # waveform shape: [channels, num_samples]
    waveform, orig_sr = torchaudio.load(path)
    print(f"Loaded:      {path}")
    print(f"  Channels:  {waveform.shape[0]}")
    print(f"  Samples:   {waveform.shape[1]}")
    print(f"  Orig SR:   {orig_sr} Hz")
    print(f"  Duration:  {waveform.shape[1] / orig_sr:.3f}s")

    # --- Resample if needed ---
    if orig_sr != target_sr:
        resampler = T.Resample(orig_freq=orig_sr, new_freq=target_sr)
        waveform = resampler(waveform)
        print(f"  Resampled: {orig_sr} Hz -> {target_sr} Hz "
              f"({waveform.shape[1]} samples)")

    # --- Convert stereo to mono (average channels) ---
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
        print(f"  Mono:      averaged {waveform.shape[0]} channels")

    print(f"  Final shape: {waveform.shape}  (1 x {waveform.shape[1]} samples)")
    return waveform, target_sr


# Create a test WAV at 44.1 kHz stereo to demonstrate the full pipeline
import soundfile as sf
import numpy as np, os
os.makedirs("audio_samples", exist_ok=True)

sr_high = 44100
duration = 2.0
t = np.linspace(0, duration, int(sr_high * duration), endpoint=False)
stereo = np.stack([
    0.3 * np.sin(2 * np.pi * 440 * t),   # left channel: 440 Hz
    0.3 * np.sin(2 * np.pi * 523 * t),   # right channel: 523 Hz
], axis=1).astype(np.float32)
sf.write("audio_samples/stereo_44k.wav", stereo, sr_high)
print("Created stereo 44.1 kHz test file.\n")

waveform_16k, sr = load_and_resample("audio_samples/stereo_44k.wav")

### Waveform and Log-Mel Spectrogram Visualization

Visualizing audio gives intuition for what the model actually sees. The waveform shows amplitude over time; the spectrogram shows frequency content over time. Whisper's feature extractor produces the log-Mel spectrogram on the right.

In [ ]:
def plot_waveform_and_spectrogram(waveform: "torch.Tensor", sample_rate: int, title: str = "Audio"):
    """
    Plot waveform and log-Mel spectrogram side by side.

    Args:
        waveform: shape [1, num_samples] or [num_samples], float32
        sample_rate: sample rate in Hz
        title: figure title
    """
    import torch

    if waveform.dim() == 2:
        audio_np = waveform[0].numpy()
    else:
        audio_np = waveform.numpy()

    num_samples = len(audio_np)
    duration = num_samples / sample_rate
    time_axis = np.linspace(0, duration, num_samples)

    # Compute log-Mel spectrogram via torchaudio
    # Parameters match Whisper's feature extractor: 80 mel bins, 25ms window, 10ms hop
    mel_transform = T.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=400,          # 25ms window at 16kHz
        hop_length=160,     # 10ms hop
        n_mels=80,
        f_min=0.0,
        f_max=8000.0,
    )
    amplitude_to_db = T.AmplitudeToDB(stype="power", top_db=80)

    wav_tensor = torch.tensor(audio_np).unsqueeze(0)  # [1, T]
    mel = mel_transform(wav_tensor)                    # [1, 80, time_frames]
    log_mel = amplitude_to_db(mel).squeeze(0).numpy() # [80, time_frames]

    # --- Plot ---
    fig = plt.figure(figsize=(14, 4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.4], wspace=0.35)

    # Left: waveform
    ax0 = fig.add_subplot(gs[0])
    ax0.plot(time_axis, audio_np, linewidth=0.6, color="#1f77b4")
    ax0.set_xlabel("Time (s)")
    ax0.set_ylabel("Amplitude")
    ax0.set_title(f"Waveform  ({duration:.2f}s @ {sample_rate} Hz)")
    ax0.set_xlim(0, duration)
    ax0.axhline(0, color="gray", linewidth=0.5, linestyle="--")

    # Right: log-Mel spectrogram
    ax1 = fig.add_subplot(gs[1])
    img = ax1.imshow(
        log_mel,
        aspect="auto",
        origin="lower",
        cmap="magma",
        extent=[0, duration, 0, 80],
    )
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Mel bin")
    ax1.set_title("Log-Mel Spectrogram (80 bins)")
    plt.colorbar(img, ax=ax1, label="dB")

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"Spectrogram shape: {log_mel.shape}  (mel_bins x time_frames)")


# Visualize the 16 kHz mono audio we just loaded/resampled
plot_waveform_and_spectrogram(waveform_16k, TARGET_SR, title="Stereo 44.1kHz -> Mono 16kHz")

In [ ]:
import os

os.makedirs("audio_samples", exist_ok=True)

def make_sine_wav(path: str, freq_hz: float = 440.0, duration_s: float = 2.0, sr: int = 16000):
    """Write a sine-wave WAV file (useful for testing the pipeline plumbing)."""
    t = np.linspace(0, duration_s, int(sr * duration_s), endpoint=False)
    audio = 0.3 * np.sin(2 * np.pi * freq_hz * t).astype(np.float32)
    sf.write(path, audio, sr)
    return path

test_wav = make_sine_wav("audio_samples/test_tone.wav")
print(f"Created {test_wav}")

# Check what was written
data, sr = sf.read(test_wav)
print(f"Shape: {data.shape}, Sample rate: {sr} Hz, Duration: {len(data)/sr:.2f}s")

In [ ]:
# Run the pipeline on our test file
# (A sine tone has no speech, so the transcription will be empty or noise)
result = asr_pipeline(test_wav)
print("Pipeline output:")
print(result)
print()
print("Keys in output:", list(result.keys()))
print("Transcription text:", repr(result["text"]))

### Pipeline Options

The pipeline accepts several useful keyword arguments:

In [ ]:
# Transcribe with word-level timestamps
result_with_timestamps = asr_pipeline(
    test_wav,
    return_timestamps=True,
)
print("With timestamps:")
print(result_with_timestamps)

# Force the language (instead of auto-detecting)
result_forced_en = asr_pipeline(
    test_wav,
    generate_kwargs={"language": "english"},
)
print("\nForced English:")
print(result_forced_en)

## 2. Lower-Level API: WhisperProcessor + WhisperForConditionalGeneration

The pipeline is convenient, but sometimes you need more control. The lower-level API separates:
- **`WhisperProcessor`**: handles audio feature extraction (spectrogram) and text tokenization/decoding
- **`WhisperForConditionalGeneration`**: the actual model (encoder + decoder)

This is the pattern used in fine-tuning code and when you need to pass custom generation arguments.

In [ ]:
MODEL_ID = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Processor and model loaded.")
print(f"Model on: {next(model.parameters()).device}")

## 3. Loading and Preparing Audio

Whisper requires audio at **16 kHz**, mono. Most real-world audio is at 44.1 kHz or 48 kHz and may be stereo. You need to resample and convert.

Two common approaches:
1. **`librosa.load`**: loads and resamples in one call, always returns float32 in [-1, 1]
2. **`soundfile.read` + manual resample**: more control, useful when you need the original sample rate too

In [ ]:
TARGET_SR = 16000  # Whisper's required sample rate

def load_audio_for_whisper(path: str) -> np.ndarray:
    """Load an audio file, resample to 16kHz mono, return float32 numpy array."""
    # librosa.load returns (array, sample_rate)
    # mono=True converts stereo to mono by averaging channels
    audio, orig_sr = librosa.load(path, sr=TARGET_SR, mono=True)
    print(f"Loaded: {path}")
    print(f"  Original SR: {orig_sr} Hz -> target {TARGET_SR} Hz (librosa handled this)")
    print(f"  Shape: {audio.shape}, dtype: {audio.dtype}")
    print(f"  Duration: {len(audio)/TARGET_SR:.2f}s")
    return audio


def load_audio_manual_resample(path: str) -> np.ndarray:
    """Alternative: load with soundfile, then resample with librosa if needed."""
    audio, orig_sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:           # stereo -> mono
        audio = audio.mean(axis=1)
    if orig_sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=TARGET_SR)
    return audio


audio_array = load_audio_for_whisper(test_wav)

### Feature Extraction

Before the audio reaches the Whisper encoder, it is converted into a log-Mel spectrogram:
- 80 Mel frequency bins
- Short-time Fourier Transform with 25ms windows, 10ms stride
- Resulting in 100 frames per second of audio
- Padded or trimmed to a fixed 30-second context window (3000 frames)

The `WhisperProcessor` does all of this.

In [ ]:
# Extract features: audio array -> input_features tensor
inputs = processor(
    audio_array,
    sampling_rate=TARGET_SR,
    return_tensors="pt",
)
input_features = inputs.input_features.to(device)

print("input_features shape:", input_features.shape)
# Shape is [batch, n_mels, time_frames] = [1, 80, 3000]
# 3000 frames = 30 seconds at 100 frames/sec
print("  batch=1, n_mels=80, time=3000 (30s window, zero-padded if audio is shorter)")

### Running the Model

In [ ]:
with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        # language=None means auto-detect
        # task="transcribe" (vs "translate" to always output English)
    )

print("Raw predicted token IDs:")
print(predicted_ids)
print(f"Shape: {predicted_ids.shape}  (batch=1, sequence_len={predicted_ids.shape[1]})")

# Decode token IDs back to text
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
print("\nDecoded transcription:")
print(transcription)

## 4. Understanding the Output

### Tokens vs. Text

Whisper uses a byte-pair encoding (BPE) tokenizer. The raw output of `model.generate()` is a sequence of integer token IDs. Special tokens in this sequence carry meaning:

| Token | Meaning |
|---|---|
| `<|startoftranscript|>` | Beginning of output sequence |
| `<|en|>`, `<|zh|>`, etc. | Detected or forced language |
| `<|transcribe|>` | Task is transcription (not translation) |
| `<|notimestamps|>` | No timestamp tokens will be produced |
| `<|endoftext|>` | End of sequence |

When you call `processor.batch_decode(..., skip_special_tokens=True)`, all of these are stripped and you get just the text.

In [ ]:
# Decode WITHOUT skipping special tokens to see the full sequence
transcription_with_specials = processor.batch_decode(predicted_ids, skip_special_tokens=False)
print("With special tokens:")
print(transcription_with_specials)

# Decode individual token IDs
print("\nFirst 10 token IDs and their decoded values:")
for token_id in predicted_ids[0][:10].tolist():
    token_text = processor.tokenizer.decode([token_id])
    print(f"  {token_id:6d}  ->  {repr(token_text)}")

### Language Detection

When no language is forced, Whisper predicts the language from the first 30 seconds of audio. You can read the detected language directly from the generated token IDs.

In [ ]:
def detect_language(audio_array: np.ndarray, processor, model, device) -> str:
    """Run Whisper's language identification on an audio clip."""
    inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        # Generate only the language token (stop after that)
        # We do this by using the model's detect_language method if available
        encoder_output = model.encoder(input_features)
        logits = model(
            input_features,
            decoder_input_ids=torch.tensor(
                [[model.config.decoder_start_token_id]], device=device
            ),
        ).logits

    # The language tokens are in a specific range in the vocabulary
    # processor.tokenizer.all_language_tokens gives us those IDs
    lang_token_ids = processor.tokenizer.all_language_tokens
    lang_logits = logits[0, 0, lang_token_ids]
    best_lang_idx = lang_logits.argmax().item()
    best_lang_token_id = lang_token_ids[best_lang_idx]
    lang_code = processor.tokenizer.decode([best_lang_token_id]).strip("<|>")
    return lang_code

# Test on our sine tone (will likely detect a random language since there's no speech)
detected = detect_language(audio_array, processor, model, device)
print(f"Detected language: {detected} (a tone with no speech gives an arbitrary result)")

### Timestamps

Whisper can produce timestamps at the chunk or word level. When using the pipeline, pass `return_timestamps=True`:

In [ ]:
# Chunk-level timestamps: each segment gets a (start, end) time in seconds
result_chunks = asr_pipeline(
    test_wav,
    return_timestamps=True,
)
print("Chunk-level output:")
print(result_chunks)
print()

# Word-level timestamps (more compute)
result_words = asr_pipeline(
    test_wav,
    return_timestamps="word",
)
print("Word-level output:")
print(result_words)

## 5b. Long-Audio Transcription with Chunking

Whisper's context window is exactly 30 seconds. Audio longer than that must be split into overlapping chunks. The pipeline handles this automatically when you pass `chunk_length_s`. The `stride_length_s` parameter controls the overlap between adjacent chunks, which helps avoid cut words at boundaries.

In [ ]:
import numpy as np
import soundfile as sf
import os

# Create a 90-second synthetic audio file to demonstrate chunking
long_sr = 16000
long_duration = 90.0
t_long = np.linspace(0, long_duration, int(long_sr * long_duration), endpoint=False)
# Sweep through several frequencies to make it slightly interesting
freqs = [220, 330, 440, 550, 660]
long_audio = sum(0.1 * np.sin(2 * np.pi * f * t_long) for f in freqs).astype(np.float32)
long_path = "audio_samples/long_tone_90s.wav"
sf.write(long_path, long_audio, long_sr)
print(f"Created {long_path} ({long_duration:.0f}s)")

# Transcribe with chunking
# chunk_length_s=30  -- process 30s chunks (matches Whisper's window)
# stride_length_s=5  -- 5s overlap on each side of a chunk boundary
# return_timestamps=True -- required for chunked transcription to stitch segments
result_chunked = asr_pipeline(
    long_path,
    chunk_length_s=30,
    stride_length_s=5,
    return_timestamps=True,
)

print(f"\nFull transcription text:\n{repr(result_chunked['text'])}\n")
print(f"Number of segments: {len(result_chunked.get('chunks', []))}")
print("\nSegments:")
for chunk in result_chunked.get("chunks", []):
    start, end = chunk["timestamp"]
    print(f"  [{start:6.2f}s - {end:6.2f}s]  {repr(chunk['text'])}")

### Word-Level Timestamp Extraction

Passing `return_timestamps="word"` gives per-word timing instead of per-segment. This is useful for subtitle generation or for aligning transcripts to audio for downstream tasks.

In [ ]:
# Word-level timestamps: return_timestamps="word"
# Note: word timestamps require a model that supports them (whisper-base and larger do)
result_word_ts = asr_pipeline(
    test_wav,
    return_timestamps="word",
)

print("Full text:", repr(result_word_ts["text"]))
print()

words = result_word_ts.get("chunks", [])
if words:
    print(f"{'Word':<20} {'Start':>8} {'End':>8}")
    print("-" * 40)
    for w in words:
        start, end = w["timestamp"]
        # start/end may be None for the last word if the model is uncertain
        start_str = f"{start:.3f}s" if start is not None else "  None"
        end_str   = f"{end:.3f}s"   if end   is not None else "  None"
        print(f"{repr(w['text']):<20} {start_str:>8} {end_str:>8}")
else:
    print("(No word-level chunks returned for this audio - expected for a synthetic tone.)")
    print("With real speech, each word gets a (start, end) timestamp pair.")

# Show the raw output structure
print("\nRaw output keys:", list(result_word_ts.keys()))
print("First chunk (if any):", result_word_ts["chunks"][:2] if result_word_ts.get("chunks") else "none")

### Language Detection: Force vs. Auto-Detect

Whisper reads the first 30 seconds of audio and emits a language token before the transcript. You can let it auto-detect, or force a specific language. The detected language appears in the output under the `language` key (when using the low-level API) or is encoded in the token stream.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch

MODEL_LANG_ID = "openai/whisper-base"
_proc = WhisperProcessor.from_pretrained(MODEL_LANG_ID)
_model = WhisperForConditionalGeneration.from_pretrained(MODEL_LANG_ID)
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_model = _model.to(_device).eval()

def detect_language_probs(audio_path: str, top_n: int = 5):
    """
    Run Whisper's language identification and return the top-N candidates
    with their probabilities.

    The language is identified from the first 30 seconds of audio.
    """
    import librosa
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    inputs = _proc(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(_device)

    with torch.no_grad():
        # Pass only the BOS token so the model predicts what comes next
        # (which is the language token)
        out = _model(
            input_features,
            decoder_input_ids=torch.tensor(
                [[_model.config.decoder_start_token_id]], device=_device
            ),
        )

    # Isolate logits for the language token positions
    lang_token_ids = _proc.tokenizer.all_language_tokens     # list of token IDs
    lang_logits = out.logits[0, 0, lang_token_ids]           # shape: [n_langs]
    probs = lang_logits.softmax(dim=-1)

    top_probs, top_idx = probs.topk(top_n)
    results = []
    for prob, idx in zip(top_probs.tolist(), top_idx.tolist()):
        token_id = lang_token_ids[idx]
        lang_code = _proc.tokenizer.decode([token_id]).strip("<|>")
        results.append((lang_code, prob))
    return results


# Auto-detect on our test tone
print("=== Auto-detect (sine tone, no real speech) ===")
lang_probs = detect_language_probs(test_wav)
for lang, prob in lang_probs:
    print(f"  {lang:<12}  {prob*100:5.1f}%")

# Forced language via the pipeline generate_kwargs
print("\n=== Forced language demo ===")
for lang in ["english", "french", "spanish"]:
    r = asr_pipeline(test_wav, generate_kwargs={"language": lang})
    print(f"  Forced {lang:<10}: {repr(r['text'])}")

print("\nNote: with real speech audio, auto-detect is usually reliable for")
print("major languages. Forcing is useful when you know the language in advance.")

## 6. Model Size Comparison: tiny vs. base vs. large-v3

Whisper comes in five sizes. Larger models are more accurate but slower. This cell loads three of them, transcribes the same audio, and measures latency and WER against a known reference.

In [ ]:
# pip install jiwer  # uncomment if needed
import time
import jiwer
from transformers import pipeline as hf_pipeline

# We use a synthetic reference string here.
# With real speech audio replace REFERENCE and AUDIO_PATH with your own files.
AUDIO_PATH = test_wav   # synthetic tone -- WER will be high; illustrates the pipeline
REFERENCE  = ""         # expected transcription; empty string for the sine tone

# Models to compare (comment out large-v3 if you are on CPU to save time)
MODELS_TO_COMPARE = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    # "openai/whisper-large-v3",   # uncomment on GPU
]

_device_id = 0 if torch.cuda.is_available() else -1
results_table = []

for model_id in MODELS_TO_COMPARE:
    print(f"Loading {model_id} ...", end=" ", flush=True)
    pipe = hf_pipeline(
        "automatic-speech-recognition",
        model=model_id,
        device=_device_id,
    )
    params_m = sum(p.numel() for p in pipe.model.parameters()) / 1e6

    # Warm-up pass
    _ = pipe(AUDIO_PATH)

    # Timed pass
    t_start = time.perf_counter()
    result = pipe(AUDIO_PATH, generate_kwargs={"language": "english"})
    latency_s = time.perf_counter() - t_start

    hyp = result["text"].strip()

    # Compute WER (only meaningful with real speech and a real reference)
    if REFERENCE:
        transform = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
                                   jiwer.Strip(), jiwer.ReduceToListOfListOfWords()])
        wer_score = jiwer.wer(REFERENCE, hyp,
                               reference_transform=transform,
                               hypothesis_transform=transform)
    else:
        wer_score = float("nan")   # no reference available

    results_table.append({
        "model": model_id.split("/")[-1],
        "params_M": round(params_m, 0),
        "latency_s": round(latency_s, 3),
        "WER": round(wer_score, 4) if not (wer_score != wer_score) else "n/a",
        "text": hyp[:60],
    })
    print(f"done  ({latency_s:.2f}s)")
    del pipe   # free memory

print()
print(f"{'Model':<22} {'Params':>8} {'Latency':>10} {'WER':>8}  {'Transcription'}")
print("-" * 90)
for row in results_table:
    print(f"{row['model']:<22} {row['params_M']:>6.0f}M {row['latency_s']:>9.3f}s "
          f"{str(row['WER']):>8}  {repr(row['text'])}")

## 5. Hands-On: Transcribing in Multiple Languages

Since we can't download real speech audio in this notebook, we'll simulate the multilingual workflow by creating synthetic audio files and demonstrating the full pipeline code you'd use with real files.

The key parameters that change between languages:
- **Auto-detect**: don't pass `language` to `generate_kwargs`
- **Force English**: `generate_kwargs={"language": "english"}`
- **Force Spanish**: `generate_kwargs={"language": "spanish"}`
- **Translate to English**: `generate_kwargs={"task": "translate"}` (always outputs English text regardless of input language)

Language codes accepted by Whisper: `english`, `spanish`, `french`, `german`, `chinese`, `japanese`, `arabic`, `portuguese`, `russian`, `hindi`, and [97 others](https://github.com/openai/whisper/blob/main/whisper/tokenizer.py).

In [ ]:
def transcribe(audio_path: str, language: str = None, task: str = "transcribe") -> dict:
    """
    Transcribe an audio file using Whisper.

    Args:
        audio_path: Path to a WAV/MP3/FLAC file.
        language: Language code (e.g. 'english', 'spanish'). None = auto-detect.
        task: 'transcribe' keeps the original language; 'translate' outputs English.

    Returns:
        Dict with 'text', 'language', and 'chunks' (if timestamps requested).
    """
    generate_kwargs = {"task": task}
    if language is not None:
        generate_kwargs["language"] = language

    result = asr_pipeline(
        audio_path,
        generate_kwargs=generate_kwargs,
        return_timestamps=True,
    )
    return result


# Demonstrate with our test tone (no real speech, but shows the API)
print("=== Auto-detect language ===")
r1 = transcribe(test_wav)
print(f"Text: {repr(r1['text'])}")

print("\n=== Forced English ===")
r2 = transcribe(test_wav, language="english")
print(f"Text: {repr(r2['text'])}")

print("\n=== Translate task (output always English) ===")
r3 = transcribe(test_wav, task="translate")
print(f"Text: {repr(r3['text'])}")

In [ ]:
# Batch inference: process multiple files at once
# When you have multiple audio files, passing them as a list is faster
# than calling the pipeline in a loop (the pipeline batches them internally)

# Create two more test audio files with different tones
tone1 = make_sine_wav("audio_samples/tone_220hz.wav", freq_hz=220.0, duration_s=3.0)
tone2 = make_sine_wav("audio_samples/tone_880hz.wav", freq_hz=880.0, duration_s=3.0)

results = asr_pipeline(
    [test_wav, tone1, tone2],
    generate_kwargs={"language": "english"},
)

for path, result in zip([test_wav, tone1, tone2], results):
    print(f"{os.path.basename(path):25s} -> {repr(result['text'])}")

### Using Real Audio Files

To use real speech audio, drop `.wav`, `.mp3`, or `.flac` files into `audio_samples/` and replace the paths in the cells above. For example:

```python
# Transcribe a real English clip
en_result = transcribe("audio_samples/english_speech.wav", language="english")
print(en_result["text"])

# Transcribe a real Spanish clip (auto-detect language)
es_result = transcribe("audio_samples/spanish_speech.wav")
print(es_result["text"])

# Translate Spanish speech to English
es_translated = transcribe("audio_samples/spanish_speech.wav", task="translate")
print(es_translated["text"])
```

Common free sources for short multilingual speech clips:
- [Mozilla Common Voice](https://commonvoice.mozilla.org/) (validated clips in 100+ languages)
- [LibriSpeech](https://www.openslr.org/12) (English read speech)

## Exercise

1. **Compare pipeline vs. low-level API**: Transcribe the same audio file using both the `pipeline` approach and the `WhisperProcessor + model.generate()` approach. Verify the text outputs are identical.

2. **Model size comparison**: Whisper comes in five sizes: `tiny`, `small`, `medium`, `large-v2`, `large-v3`. Load `openai/whisper-tiny` and `openai/whisper-small`, run both on the same audio clip, and measure the inference time with `time.time()`. How much does size affect speed?

3. **Timestamp parsing**: When `return_timestamps=True`, the output contains a `chunks` key with a list of `{"text": ..., "timestamp": (start, end)}` dicts. Write a function that takes this output and formats it as a subtitle file (SRT format):
   ```
   1
   00:00:00,000 --> 00:00:02,500
   Hello, this is the first segment.
   ```

4. **(Stretch)** Force Whisper to transcribe the same audio in three different languages (pass `language="english"`, `language="spanish"`, `language="french"`). What happens? Does it hallucinate text in those languages? This is a known Whisper behavior called "language hallucination" and it's one of the things fine-tuning helps address.

## Exercise: Download and Transcribe a Common Voice Clip

Download a short audio clip from Common Voice using the `datasets` library, transcribe it with `whisper-base`, and print the model output alongside the reference transcript.

In [ ]:
# pip install datasets soundfile  # uncomment if needed
# Note: Common Voice requires a free HuggingFace account and accepting dataset terms.
# Run `huggingface-cli login` in a terminal once, then re-run this cell.

# YOUR CODE HERE
# Steps:
#   1. Load a single example from Common Voice English validation split:
#        from datasets import load_dataset, Audio
#        cv = load_dataset("mozilla-foundation/common_voice_13_0", "en",
#                          split="validation", streaming=True, trust_remote_code=False)
#        cv = cv.cast_column("audio", Audio(sampling_rate=16000))
#        example = next(iter(cv))
#
#   2. Save the audio array to a temporary WAV file with soundfile.
#
#   3. Transcribe the saved WAV with `asr_pipeline` (whisper-base loaded above).
#
#   4. Print:
#        - The reference sentence: example["sentence"]
#        - The model transcription: result["text"]
#
#   Bonus: compute the WER between the reference and the hypothesis using jiwer.

raise NotImplementedError